# Base Model Smoke Test

Loads **Llama-3.2-3B-Instruct** in 4-bit (bitsandbytes) on a Colab T4, runs the fixed set of
Salesforce dev prompts from `data/baseline_prompts.json`, and saves the outputs as the
"before fine-tuning" reference for later evaluation (Epic D).

**Before running:** Llama-3.2 is a gated model on Hugging Face. Request access at
https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and create a token with
`Read` access, then add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets sentencepiece

In [ ]:
import time
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def generate(instruction: str, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = output_ids[0][inputs.shape[-1]:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

Upload `data/baseline_prompts.json` from the repo into this Colab session's working directory
(left sidebar -> Files -> upload) before running the next cell.

In [ ]:
with open("baseline_prompts.json") as f:
    prompts = json.load(f)

results = []
for p in prompts:
    print(f"Running: {p['id']}")
    t0 = time.time()
    response = generate(p["instruction"])
    gen_time = time.time() - t0
    results.append({
        "id": p["id"],
        "category": p["category"],
        "instruction": p["instruction"],
        "base_model_response": response,
        "gen_time_seconds": round(gen_time, 1),
    })
    print(response[:300], "...\n---")

In [ ]:
with open("baseline_responses.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved baseline_responses.json")
print(f"\nModel: {MODEL_ID}")
print(f"Load time: {load_time:.1f}s")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Notes

After running, copy `baseline_responses.json` into `eval/results/` in the repo (this becomes
the "before fine-tuning" reference for Story D1), and fill in below:

- **Load time:** _TBD_
- **GPU memory allocated:** _TBD_
- **Generation time per prompt (avg):** _TBD_
- **Any compatibility issues encountered:** _TBD_